# Introducción a API tf.data

##  Desarrollado por:

Luis Octavio Gonzalez Salcedo

Diplomado en Inteligencia Artificial & Aprendizaje Profundo

## Introducción


Tomado de [[1]](https://www.tensorflow.org/guide/data?hl=en)

En el presente cuaderno se introduce el concepto de **API tf.data** y los diferentes procesos que involucran el manejo y operaciones con esta herramienta. Su término en inglés corresponde a tf.data API

La API tf.data permite crear tuberías de entrada complejas a partir de piezas simples y reutilizables. Por ejemplo, la canalización de un modelo de imagen podría agregar datos de archivos en un sistema de archivos distribuido, aplicar perturbaciones aleatorias a cada imagen y fusionar imágenes seleccionadas al azar en un lote para entrenamiento. La canalización de un modelo de texto puede implicar extraer símbolos de datos de texto sin procesar, convertirlos en identificadores incrustados con una tabla de búsqueda y agrupar secuencias de diferentes longitudes. La API tf.data permite manejar grandes cantidades de datos, leer desde diferentes formatos de datos y realizar transformaciones complejas.

La API tf.data introduce una tf.data.Dataset, abstracción que representa una secuencia de elementos, en la que cada elemento consta de uno o más componentes. Por ejemplo, en una canalización de imágenes, un elemento puede ser un solo ejemplo de entrenamiento, con un par de componentes tensoriales que representan la imagen y su etiqueta.

Hay dos formas distintas de crear un conjunto de datos:

    Una fuente de datos construye a Dataset a partir de los datos almacenados en la memoria o en uno o más archivos.

    Una transformación de datos construye un conjunto de datos a partir de uno o más tf.data.Dataset objetos.



## Importación de librerías

Se requieren importar las siguientes librerías:

**TensorFlow**: librería Python para computación numérica rápida.  También es una librería que puede utilizarse para crear modelos de Deep Learning directamente o utilizando librerías de envolturas que simplifican el proceso construido sobre TensorFlow.

**NumPy**: biblioteca para el lenguaje de programación Python que da soporte para crear vectores y matrices grandes multidimensionales, junto con una gran colección de funciones matemáticas de alto nivel para operar con ellas.

**pathlib**: Es un módulo que ofrece clases que representan rutas del sistema de archivos con semántica apropiada para diferentes sistemas operativos. Las clases de ruta se dividen entre rutas puras , que proporcionan operaciones puramente computacionales sin Entradas / Salidas, y rutas concretas, que heredan de rutas puras pero también proporcionan operaciones de Entrada / Salida. Crea una instancia de una ruta concreta para la plataforma en la que se ejecuta el código. Los caminos puros son útiles en algunos casos especiales; por ejemplo: 1.- Si desea manipular las rutas de Windows en una máquina Unix (o viceversa). No puede crear una instancia de un WindowsPathcuando se ejecuta en Unix, pero puede crear una instancia PureWindowsPath; 2.- Desea asegurarse de que su código solo manipule rutas sin acceder al sistema operativo. En este caso, la instanciación de una de las clases puras puede ser útil, ya que simplemente no tienen operaciones de acceso al sistema operativo. [[2]](https://docs.python.org/3/library/pathlib.html).

**os**: Este módulo os proporciona un contenedor para módulos específicos de la plataforma, tales como posix, nt y mac. La interfaz de programación para las funciones disponibles en todas las plataformas debe ser la misma, por lo que usar el módulo os ofrece cierta medida de portabilidad. Sin embargo, no todas las funciones están disponibles en todas las plataformas. Muchas de las funciones de gestión de procesos descritas no están disponibles para Windows. [[3]](https://rico-schmidt.name/pymotw-3/os/).

**pandas**:  Pandas es una herramienta de manipulación y análisis de datos de código abierto rápida, potente, flexible y fácil de usar, construida sobre el lenguaje de programación Python . [[4]](https://pandas.pydata.org/).


**matplotlib**: Matplotlib es una biblioteca completa para crear visualizaciones estáticas, animadas e interactivas en Python. [[5]](https://matplotlib.org/).


In [2]:

import tensorflow as tf ##importación de tensorflow

import pathlib ##importación del módulo pathlib
import os ##importación del módulo os
import matplotlib.pyplot as plt ##importación de matplotlib
import pandas as pd ##importación de pandas
import numpy as np ##importación de numpy

##configura las opciones de impresión
##Estas opciones determinan la forma en que se muestran los números de punto flotante, las matrices y otros objetos NumPy.
##En este caso, usa el parámetro de precisión, que corresponde al número de dígitos de precisión para salida de punto flotante.
np.set_printoptions(precision=4)


## Apectos Esenciales

Para crear una canalización de entrada, debe comenzar con una fuente de datos. Por ejemplo, para construir un Dataset de datos a partir de datos en la memoria, puede usar tf.data.Dataset.from_tensors() o tf.data.Dataset.from_tensor_slices(). Alternativamente, si sus datos de entrada están almacenados en un archivo en el formato TFRecord de TensorFlow puede usar tf.data.TFRecordDataset().

El objeto Dataset es un iterable de Python. Esto hace posible consumir sus elementos usando un bucle for:

In [3]:
dataset = tf.data.Dataset.from_tensor_slices([8, 3, 0, 8, 2, 1])
dataset

<TensorSliceDataset shapes: (), types: tf.int32>

In [4]:


len(dataset)



6

In [5]:
for elem in dataset:
    print(elem.numpy())

8
3
0
8
2
1


o se pueden crear explícitamente un iterador

In [6]:


it = iter(dataset)
print(next(it).numpy())



8



Alternativamente, los elementos del conjunto de datos se pueden consumir utilizando la transformación de reduce, que reduce todos los elementos para producir un único resultado


## Estructura del conjunto de datos


Un conjunto de datos produce una secuencia de elementos , donde cada elemento tiene la misma estructura (anidada) de componentes .

Los componentes individuales de la estructura pueden ser de cualquier tipo representable por tf.TypeSpec, incluidos tf.Tensor, tf.sparse.SparseTensor, tf.RaggedTensor, tf.TensorArray o tf.data.Dataset.

Las construcciones de Python que se pueden usar para expresar la estructura (anidada) de elementos incluyen tuple, dict, NamedTuple y OrderedDict.

En particular, list no es una construcción válida para expresar la estructura de los elementos del conjunto de datos. Esto se debe a que los primeros usuarios de tf.data estaban convencidos de que las entradas de list (por ejemplo, pasadas a tf.data.Dataset.from_tensors ) se empaquetaran automáticamente como tensores y las salidas de list (por ejemplo, valores de retorno de funciones definidas por el usuario) se coaccionaran en una tuple.

Debido a la argumentación anterior, si desea que una entrada de list se trate como una estructura, debe convertirla en tuple y si desea que una lista de salida, entonces debe empaquetarla explícitamente usando tf.stack.

La propiedad Dataset.element_spec permite inspeccionar el tipo de cada componente del elemento. La propiedad devuelve una estructura anidada de objetos tf.TypeSpec, que coincide con la estructura del elemento, que puede ser un solo componente, una tupla de componentes o una tupla anidada de componentes. Por ejemplo:


In [7]:
dataset1 = tf.data.Dataset.from_tensor_slices(tf.random.uniform([4, 10]))
print(dataset1)

<TensorSliceDataset shapes: (10,), types: tf.float32>


In [8]:
for i in dataset1:
    print(i.numpy())

[0.3815 0.4921 0.9272 0.1137 0.0152 0.5072 0.28   0.1549 0.563  0.9648]
[0.7519 0.5754 0.1009 0.5748 0.5742 0.9504 0.3312 0.664  0.5789 0.7463]
[0.2819 0.4741 0.8478 0.9033 0.0818 0.051  0.4876 0.2668 0.7397 0.9065]
[0.37   0.548  0.3192 0.7391 0.2031 0.4857 0.4786 0.1329 0.0132 0.4387]


In [9]:
len(dataset1)

4

In [10]:
dataset2 = tf.data.Dataset.from_tensor_slices(
    (tf.random.uniform([4]),
     tf.random.uniform([4,100], maxval=100, dtype=tf.int32)))

dataset2.element_spec

(TensorSpec(shape=(), dtype=tf.float32, name=None),
 TensorSpec(shape=(100,), dtype=tf.int32, name=None))

In [11]:
len(dataset2)

4

In [12]:
dataset3 = tf.data.Dataset.zip((dataset1, dataset2))
dataset3.element_spec

(TensorSpec(shape=(10,), dtype=tf.float32, name=None),
 (TensorSpec(shape=(), dtype=tf.float32, name=None),
  TensorSpec(shape=(100,), dtype=tf.int32, name=None)))

In [13]:
len(dataset3)

4

In [14]:
type(dataset3)

tensorflow.python.data.ops.dataset_ops.ZipDataset

In [15]:
i = iter(dataset3)

In [16]:
print(i.next(), "\n")
print(i.next(), "\n")
print(i.next(), "\n")
print(i.next(), "\n")

(<tf.Tensor: shape=(10,), dtype=float32, numpy=
array([0.3815, 0.4921, 0.9272, 0.1137, 0.0152, 0.5072, 0.28  , 0.1549,
       0.563 , 0.9648], dtype=float32)>, (<tf.Tensor: shape=(), dtype=float32, numpy=0.3305838>, <tf.Tensor: shape=(100,), dtype=int32, numpy=
array([37, 12, 10, 63,  2, 61, 60, 19, 44, 31,  6, 57, 58, 26,  3, 69, 49,
       31, 75,  3, 22,  2, 27, 64, 57, 39, 47, 28, 70, 15, 26, 58, 44, 43,
       19, 36, 68, 96, 44, 86, 34, 37, 34, 92, 56, 45,  7, 89, 95, 83, 75,
       76, 57, 12, 33, 61, 56, 71, 38, 47, 56, 71, 63, 91, 94, 79, 69, 80,
        0, 38, 54, 51, 33, 40, 16, 61, 97, 77, 50, 76, 51, 49, 56, 73, 62,
       12, 20, 25, 35, 88, 14, 41, 97, 56, 42, 22, 56, 59, 92, 18])>)) 

(<tf.Tensor: shape=(10,), dtype=float32, numpy=
array([0.7519, 0.5754, 0.1009, 0.5748, 0.5742, 0.9504, 0.3312, 0.664 ,
       0.5789, 0.7463], dtype=float32)>, (<tf.Tensor: shape=(), dtype=float32, numpy=0.35826302>, <tf.Tensor: shape=(100,), dtype=int32, numpy=
array([88, 22, 71,  1, 89, 

In [17]:
# dataset con tensores dispersos
dataset4 = tf.data.Dataset.from_tensors(tf.SparseTensor(indices=[[0, 0],[1, 2]], values=[1, 2], dense_shape=[3, 4]))
dataset4.element_spec

SparseTensorSpec(TensorShape([3, 4]), tf.int32)

In [18]:
dataset4.element_spec.value_type

tensorflow.python.framework.sparse_tensor.SparseTensor

In [19]:


for a, (b,c) in dataset3:
    print('shapes: {a.shape}, {b.shape}, {c.shape}'.format(a=a, b=b, c=c))



shapes: (10,), (), (100,)
shapes: (10,), (), (100,)
shapes: (10,), (), (100,)
shapes: (10,), (), (100,)



Los Dataset transformaciones soportan los conjuntos de datos de cualquier estructura. Cuando se utilizan las Dataset.map() y Dataset.filter() , que aplican una función a cada elemento, la estructura del elemento determina los argumentos de la función:


In [20]:

dataset1 = tf.data.Dataset.from_tensor_slices(
    tf.random.uniform([4, 10], minval=1, maxval=10, dtype=tf.int32))

dataset1


<TensorSliceDataset shapes: (10,), types: tf.int32>

In [21]:

for z in dataset1:
  print(z.numpy())


[7 3 7 2 1 4 6 5 1 1]
[2 3 4 5 5 9 6 7 8 9]
[1 7 6 2 5 9 6 4 5 9]
[5 3 7 7 5 5 1 9 9 9]


In [22]:

dataset2 = tf.data.Dataset.from_tensor_slices(
   (tf.random.uniform([4]),
    tf.random.uniform([4, 100], maxval=100, dtype=tf.int32)))

dataset2


<TensorSliceDataset shapes: ((), (100,)), types: (tf.float32, tf.int32)>

In [23]:

dataset3 = tf.data.Dataset.zip((dataset1, dataset2))

dataset3


<ZipDataset shapes: ((10,), ((), (100,))), types: (tf.int32, (tf.float32, tf.int32))>

In [24]:

for a, (b,c) in dataset3:
  print('shapes: {a.shape}, {b.shape}, {c.shape}'.format(a=a, b=b, c=c))


shapes: (10,), (), (100,)
shapes: (10,), (), (100,)
shapes: (10,), (), (100,)
shapes: (10,), (), (100,)



## Leer datos de entrada


### Consumir matrices Numpy



Si todos sus datos de entrada caben en la memoria, la forma más sencilla de crear un Dataset a partir de ellos es convertirlos en objetos tf.Tensor y usar Dataset.from_tensor_slices() .


In [25]:


train, test = tf.keras.datasets.fashion_mnist.load_data()



In [26]:
imagenes, labels  = train
imagenes = imagenes /255.

dataset = tf.data.Dataset.from_tensor_slices((imagenes, labels))
dataset

<TensorSliceDataset shapes: ((28, 28), ()), types: (tf.float64, tf.uint8)>

*Nota: Se encuentra la siguiente anotación sobre el fragmento del código anterior " El fragmento de código anterior incorporará las matrices de features y labels en tu gráfico de TensorFlow como operaciones tf.constant() . Esto funciona bien para un conjunto de datos pequeño, pero desperdicia memoria --- porque el contenido de la matriz se copiará varias veces --- y puede tf.GraphDef el límite de 2GB para el tf.GraphDef protocolo tf.GraphDef ."*


### Consumir generadores de Python

Otra fuente de datos común que se puede ingerir fácilmente comotf.data.Dataset es el generador de Python.


In [27]:
def count(stop):
    i=0
    while i<stop:
        yield i
        i+= 1
        
for n in count(5):
    print(n)

0
1
2
3
4




El constructor Dataset.from_generator convierte el generador de Python en un tf.data.Dataset completamente funcional.

El constructor toma un invocable como entrada, no un iterador. Esto le permite reiniciar el generador cuando llega al final. Toma un argumento args opcional, que se pasa como argumentos del invocable.

El argumento output_types es necesario porque tf.data crea un tf.Graph internamente y los bordes del gráfico requieren un tf.dtype .


In [28]:
ds_counter = tf.data.Dataset.from_generator(count, args=[25], output_types=tf.int32, output_shapes=(),)

In [29]:
for count_batch in ds_counter.repeat().batch(10).take(10):
    print(count_batch.numpy())

[0 1 2 3 4 5 6 7 8 9]
[10 11 12 13 14 15 16 17 18 19]
[20 21 22 23 24  0  1  2  3  4]
[ 5  6  7  8  9 10 11 12 13 14]
[15 16 17 18 19 20 21 22 23 24]
[0 1 2 3 4 5 6 7 8 9]
[10 11 12 13 14 15 16 17 18 19]
[20 21 22 23 24  0  1  2  3  4]
[ 5  6  7  8  9 10 11 12 13 14]
[15 16 17 18 19 20 21 22 23 24]




El argumento output_shapes no es necesario, pero se recomienda, ya que muchas operaciones de flujo tensorial no admiten tensores con rango desconocido. Si la longitud de un eje en particular es desconocida o variable, output_shapes puede colcarse como None en output_shapes.

También es importante tener en cuenta que output_shapes y output_types siguen las mismas reglas de anidamiento que otros métodos de conjuntos de datos.

Aquí hay un generador de ejemplo que demuestra ambos aspectos, devuelve tuplas de matrices, donde la segunda matriz es un vector con longitud desconocida.


In [30]:


def gen_series():
    i = 0
    while True:
        size = np.random.randint(0,10)
        yield i, np.random.normal(size = (size,))
        i+=1



In [31]:
for i, series in gen_series():
    print(i, ":", str(series))
    if i > 5:
        break

0 : [-0.4305  0.3398]
1 : []
2 : [ 1.2032 -0.8117  1.0107  0.3015  0.3695 -0.2717 -1.4779 -0.5883]
3 : [ 0.2799 -1.6661]
4 : [-0.6539 -1.6184  0.4465  2.2015 -1.1095 -0.0764  1.101   1.0393  0.8324]
5 : []
6 : [ 1.0266 -0.1505 -0.6916 -0.1929 -0.6426 -0.401  -0.3349 -2.4464 -0.305 ]




La primera salida es un int32 la segunda es un float32.

El primer elemento es un escalar, forma () , y el segundo es un vector de longitud desconocida, forma (None,)


In [32]:


ds_series = tf.data.Dataset.from_generator(
    gen_series,
    output_types=(tf.int32, tf.float32),
    output_shapes=((), (None, )))

ds_series



<FlatMapDataset shapes: ((), (None,)), types: (tf.int32, tf.float32)>

Ahora se puede utilizar como un tf.data.Dataset normal. Tenga en cuenta que al procesar por lotes un conjunto de datos con una forma variable, debe usar Dataset.padded_batch.

In [33]:
ds_series_batch = ds_series.shuffle(20).padded_batch(10)

ids, sequence_batch = next(iter(ds_series_batch))

print (ids.numpy())
print()
print(sequence_batch.numpy())

[ 1 20 19  6  7 16 24  3 14 28]

[[ 0.0157 -0.0771  1.2789  0.6987  0.      0.      0.      0.      0.    ]
 [-1.0013 -0.0759 -0.0189 -1.2184 -1.6108  0.      0.      0.      0.    ]
 [ 1.5914 -0.0866 -0.5152 -1.4901 -0.3837  0.2317 -1.5753  0.      0.    ]
 [ 1.5521 -1.3     0.735   1.4571  0.      0.      0.      0.      0.    ]
 [-0.6834  0.7265  0.8884 -0.6355  0.      0.      0.      0.      0.    ]
 [ 0.2967 -0.6291  1.0088 -0.95   -0.9801  0.      0.      0.      0.    ]
 [ 0.      0.      0.      0.      0.      0.      0.      0.      0.    ]
 [ 0.      0.      0.      0.      0.      0.      0.      0.      0.    ]
 [ 0.1625 -0.8355  0.3236 -0.3098 -1.0328  0.5797  1.9306  2.1763  0.    ]
 [-0.4734  2.2363  0.8209 -1.6945 -0.9349  0.8463  0.926   0.4779  0.8261]]


In [34]:
it = iter(ds_series_batch)
for i in range(10):
    ids, sequence_batch = next(it)
    print (ids.numpy())
    print()
    print(sequence_batch.numpy())
    print()

[ 2 18 15  6 16  1  7 14 13 24]

[[ 0.0164 -1.3635  0.0643 -0.2011 -0.7229 -0.8568  0.      0.      0.    ]
 [ 0.3788 -0.762   2.2218  0.      0.      0.      0.      0.      0.    ]
 [ 1.2285  1.3991  1.1575 -1.4557 -1.9045  0.6845 -0.1134  0.      0.    ]
 [ 0.7211  0.5998 -0.1461  0.6161 -0.1235  0.3826  0.8484 -0.8664  0.    ]
 [ 1.3455 -0.6706 -1.5834 -1.7621  2.1226 -0.2816  0.      0.      0.    ]
 [ 0.8875 -0.0073 -0.1568 -0.4338 -0.4444 -0.9042 -0.2821 -1.2468  1.5176]
 [-0.6895  0.396   0.      0.      0.      0.      0.      0.      0.    ]
 [-1.604  -0.1845 -0.2697  0.0697  0.7873  0.      0.      0.      0.    ]
 [ 0.0514  0.      0.      0.      0.      0.      0.      0.      0.    ]
 [ 0.7856 -0.0157 -0.3312 -0.1648  0.      0.      0.      0.      0.    ]]

[23  0 10 29 33 12  9 30 28 25]

[[-0.0683 -0.6105  1.083  -1.2294  1.0957  0.0505  0.      0.    ]
 [ 0.0402 -1.4722 -1.2355 -1.7204  0.6445  0.3861  0.8888  0.    ]
 [-1.7292 -0.1479  0.0349  1.5191  0.5988 -1.381

### Ejemplo realista con imágenes



Para obtener un ejemplo más realista, intente tf.data.Dataset preprocessing.image.ImageDataGenerator como un tf.data.Dataset .

Primero descargue los datos:


In [35]:
flowers = tf.keras.utils.get_file(
    'flower_photos',
    'https://storage.googleapis.com/download.tensorflow.org/example_images/flower_photos.tgz',
    untar=True)

In [36]:


print(flowers)



C:\Users\Usuario\.keras\datasets\flower_photos




Cree la image.ImageDataGenerator


In [37]:


image_gen = tf.keras.preprocessing.image.ImageDataGenerator(rescale=1./255, rotation_range=20)



In [38]:
images, labels = next(image_gen.flow_from_directory(flowers))

Found 3670 images belonging to 5 classes.


In [39]:
print(images.dtype, images.shape)
print(labels.dtype, labels.shape)

float32 (32, 256, 256, 3)
float32 (32, 5)


In [40]:
ds = tf.data.Dataset.from_generator(
    lambda: image_gen.flow_from_directory(flowers),
    output_types=(tf.float32, tf.float32),
    output_shapes=([32,256,256,3],[32,5]))

ds.element_spec

(TensorSpec(shape=(32, 256, 256, 3), dtype=tf.float32, name=None),
 TensorSpec(shape=(32, 5), dtype=tf.float32, name=None))

### Consumir datos de TFRecord

La API tf.data admite una variedad de formatos de archivo para que pueda procesar grandes conjuntos de datos que no caben en la memoria. Por ejemplo, el formato de archivo TFRecord es un formato binario simple orientado a registros que muchas aplicaciones de TensorFlow usan para entrenar datos. La clase tf.data.TFRecordDataset permite transmitir el contenido de uno o más archivos TFRecord como parte de una canalización de entrada.

A continuación, se muestra un ejemplo que utiliza el archivo de prueba de los carteles de nombres de calles franceses (FSNS):


In [41]:

# Creates a dataset that reads all of the examples from two files.
fsns_test_file = tf.keras.utils.get_file("fsns.tfrec", "https://storage.googleapis.com/download.tensorflow.org/data/fsns-20160927/testdata/fsns-00000-of-00001")



El argumento de filenames para el inicializador TFRecordDataset puede ser una cadena, una lista de cadenas o un tf.Tensor de cadenas. Por lo tanto, si tiene dos conjuntos de archivos con fines de entrenamiento y validación, puede crear un método de fábrica que produzca el conjunto de datos, tomando los nombres de archivo como argumento de entrada:


In [42]:

dataset = tf.data.TFRecordDataset(filenames = [fsns_test_file])
dataset


<TFRecordDatasetV2 shapes: (), types: tf.string>


Muchos proyectos de TensorFlow utilizan registros tf.train.Example serializados en sus archivos TFRecord. Estos deben decodificarse antes de poder inspeccionarse:


In [43]:

raw_example = next(iter(dataset))
parsed = tf.train.Example.FromString(raw_example.numpy())

parsed.features.feature['image/text']


bytes_list {
  value: "Rue Perreyon"
}

### Consumir datos de texto

Muchos conjuntos de datos se distribuyen como uno o más archivos de texto. tf.data.TextLineDataset proporciona una manera fácil de extraer líneas de uno o más archivos de texto.

Dados uno o más nombres de archivo, un TextLineDataset producirá un elemento con valor de cadena por línea de esos archivos.


In [44]:
directory_url = 'https://storage.googleapis.com/download.tensorflow.org/data/illiad/'
file_names = ['cowper.txt', 'derby.txt', 'butler.txt']

file_paths = [
    tf.keras.utils.get_file(file_name, directory_url +file_name)
    for file_name in file_names
]

In [45]:
file_paths

['C:\\Users\\Usuario\\.keras\\datasets\\cowper.txt',
 'C:\\Users\\Usuario\\.keras\\datasets\\derby.txt',
 'C:\\Users\\Usuario\\.keras\\datasets\\butler.txt']

In [46]:
dataset = tf.data.TextLineDataset(file_paths)


Acá están las primeras líneas del primer archivo:


In [47]:
for line in dataset.take(5):
    print(line.numpy())

b"\xef\xbb\xbfAchilles sing, O Goddess! Peleus' son;"
b'His wrath pernicious, who ten thousand woes'
b"Caused to Achaia's host, sent many a soul"
b'Illustrious into Ades premature,'
b'And Heroes gave (so stood the will of Jove)'




Para alternar líneas entre archivos, use Dataset.interleave . Esto facilita la reproducción aleatoria de archivos. Aquí están la primera, segunda y tercera líneas de cada traducción:


In [48]:
file_ds = tf.data.Dataset.from_tensor_slices(file_paths)

In [49]:
for i in file_ds: print(i.numpy())

b'C:\\Users\\Usuario\\.keras\\datasets\\cowper.txt'
b'C:\\Users\\Usuario\\.keras\\datasets\\derby.txt'
b'C:\\Users\\Usuario\\.keras\\datasets\\butler.txt'


In [50]:
files_ds = tf.data.Dataset.from_tensor_slices(file_paths)
line_ds = files_ds.interleave(tf.data.TextLineDataset, cycle_length=3)

for i, line in enumerate(lines_ds.take(9)):
    if i % 3 == 0:
        print()
    print(line.numpy())


NameError: name 'lines_ds' is not defined

In [51]:


t = tf.constant([
  [[0, 1, 2, 3, 4],
   [5, 6, 7, 8, 9]],
  [[10, 11, 12, 13, 14],
   [15, 16, 17, 18, 19]],
  [[20, 21, 22, 23, 24],
   [25, 26, 27, 28, 29]],]) 
print(t.numpy())



[[[ 0  1  2  3  4]
  [ 5  6  7  8  9]]

 [[10 11 12 13 14]
  [15 16 17 18 19]]

 [[20 21 22 23 24]
  [25 26 27 28 29]]]


In [52]:
print('Extrae la primera capa\n', t[0,:,:])
print('Extrae la segunda capa\n', t[1])

Extrae la primera capa
 tf.Tensor(
[[0 1 2 3 4]
 [5 6 7 8 9]], shape=(2, 5), dtype=int32)
Extrae la segunda capa
 tf.Tensor(
[[10 11 12 13 14]
 [15 16 17 18 19]], shape=(2, 5), dtype=int32)


In [53]:
x = tf.constant([[1],[2],[3]])
print(x.shape)
print(x.shape.as_list())

(3, 1)
[3, 1]


In [54]:
reshaped = tf.reshape(x,[1,3])
print(reshaped.numpy())
print(x.numpy())

[[1 2 3]]
[[1]
 [2]
 [3]]




Con tf.reshape los datos mantienen su disposición en la memoria y se crea un nuevo tensor, con la forma solicitada, apuntando a los mismos datos. TensorFlow usa un orden de memoria de "fila principal" de estilo C, donde incrementar el índice de la derecha corresponde a un solo paso en la memoria.




### Aplana un tensor



Esta operación permite ver el orden como están organizados los datos en memoria


In [55]:
print("t actual: \n", t.numpy())
flat = tf.reshape(t, [-1])
print("\n t aplanado: \n", flat.numpy())

t actual: 
 [[[ 0  1  2  3  4]
  [ 5  6  7  8  9]]

 [[10 11 12 13 14]
  [15 16 17 18 19]]

 [[20 21 22 23 24]
  [25 26 27 28 29]]]

 t aplanado: 
 [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22 23
 24 25 26 27 28 29]




Normalmente, el único uso razonable de tf.reshape es combinar o dividir ejes adyacentes (o agregar / eliminar 1 s).

Para este tensor de 2x3x5, remodelar a (2x3)x5 o 2x (3x5) son dos cosas razonables, ya que los cortes no se mezclan:


In [56]:
print(t.shape.as_list())

[3, 2, 5]


In [57]:
print(tf.reshape(t, [2*3,5]))

tf.Tensor(
[[ 0  1  2  3  4]
 [ 5  6  7  8  9]
 [10 11 12 13 14]
 [15 16 17 18 19]
 [20 21 22 23 24]
 [25 26 27 28 29]], shape=(6, 5), dtype=int32)


In [58]:
print(tf.reshape(t, [2,3*5]))

tf.Tensor(
[[ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14]
 [15 16 17 18 19 20 21 22 23 24 25 26 27 28 29]], shape=(2, 15), dtype=int32)




No es necesario definir todos tamaños en todas las dimensiones. veámos como rehacer los dos ejemplos ateriores, respecitivamente. El -1 le dice a tf que decida cuál es la dimensión correcta.


In [59]:


print(tf.reshape(t, [-1,5]))



tf.Tensor(
[[ 0  1  2  3  4]
 [ 5  6  7  8  9]
 [10 11 12 13 14]
 [15 16 17 18 19]
 [20 21 22 23 24]
 [25 26 27 28 29]], shape=(6, 5), dtype=int32)


In [60]:
print(tf.reshape(t, [3,-1]))

tf.Tensor(
[[ 0  1  2  3  4  5  6  7  8  9]
 [10 11 12 13 14 15 16 17 18 19]
 [20 21 22 23 24 25 26 27 28 29]], shape=(3, 10), dtype=int32)




Aunque es posible hacer el siguiente reshape no tiene sentido, porque se pierde la integridad de la información.

Asegúrese que entiende la razón.


In [62]:
print(tf.reshape(t, [3,2,5]))

tf.Tensor(
[[[ 0  1  2  3  4]
  [ 5  6  7  8  9]]

 [[10 11 12 13 14]
  [15 16 17 18 19]]

 [[20 21 22 23 24]
  [25 26 27 28 29]]], shape=(3, 2, 5), dtype=int32)


Definición Conversión de tipos. Cast

In [63]:
f64_tensor = tf.constant([2.0, 4.0, 6.0], dtype = tf.float64)
print(f64_tensor)

f16_tensor = tf.cast(f64_tensor,dtype= tf.float16)
print(f16_tensor)

u8_tensor = tf.cast(f16_tensor, dtype = tf.uint8)
print(u8_tensor)

tf.Tensor([2. 4. 6.], shape=(3,), dtype=float64)
tf.Tensor([2. 4. 6.], shape=(3,), dtype=float16)
tf.Tensor([2 4 6], shape=(3,), dtype=uint8)


### Radiofusión (broadcasting)



La radiodifusión es un concepto tomado de la función equivalente en NumPy . En resumen, bajo ciertas condiciones, los tensores más pequeños se "estiran" automáticamente para adaptarse a tensores más grandes cuando se ejecutan operaciones combinadas en ellos.

El caso más simple y común es cuando intenta multiplicar o agregar un tensor a un escalar. En ese caso, el escalar se transmite para que tenga la misma forma que el otro argumento.


In [65]:
x = tf.constant([1, 2 ,3])
y = tf.constant(2)
z = tf.constant([2, 2 ,2])

# el mismo resultado
print(tf.multiply(x,2))
print(x*y)
print(x*z)

tf.Tensor([2 4 6], shape=(3,), dtype=int32)
tf.Tensor([2 4 6], shape=(3,), dtype=int32)
tf.Tensor([2 4 6], shape=(3,), dtype=int32)


In [66]:
x = tf.reshape(x, [3,1])
y = tf.range(1,5)
print(x, "\n")
print(y, "\n")

tf.Tensor(
[[1]
 [2]
 [3]], shape=(3, 1), dtype=int32) 

tf.Tensor([1 2 3 4], shape=(4,), dtype=int32) 



In [67]:
print(tf.multiply(x,y))

tf.Tensor(
[[ 1  2  3  4]
 [ 2  4  6  8]
 [ 3  6  9 12]], shape=(3, 4), dtype=int32)




Del mismo modo, los ejes con longitud 1 se pueden estirar para que coincidan con los otros argumentos. Ambos argumentos se pueden estirar en el mismo cálculo.

En este caso, una matriz de 3x1 se multiplica por elementos por una matriz de 1x4 para producir una matriz de 3x4. Observe que el 1 inicial es opcional: la forma de y es [4]. En matemáticas esta multiplicación se conoce como producto externo.



tf.broadcast_to

In [68]:
print(tf.broadcast_to([1,2,3], [3,3]))

tf.Tensor(
[[1 2 3]
 [1 2 3]
 [1 2 3]], shape=(3, 3), dtype=int32)


tf.convert_to_tensor



La mayoría de las operaciones, como tf.matmul y tf.reshape toman argumentos de la clase tf.Tensor. Sin embargo, notará que en el caso anterior, se aceptan objetos de Python con forma de tensores.

La mayoría, pero no todas, las operaciones llaman a tf.convert_to_tensor con argumentos no tensoriales. Existe un registro de conversiones, y la mayoría de las clases de objetos como ndarray , TensorShape , de Python, y tf.Variable se convertirán todas automáticamente.


### Tensores irregulares (ragged tensors) 



Un tensor con números variables de elementos a lo largo de algún eje se llama "irregular". Utilice tf.ragged.RaggedTensor para datos irregulares.

Por ejemplo, esto no se puede representar como un tensor regular:

ragged


In [69]:
ragged_list = [
    [0, 1, 2, 3],
    [4, 5],
    [6, 7, 8],
    [9]]

In [70]:
try:
    tensor = tf.constant(ragged_list)
except Exception as e:
     print(f"{type(e).__name__}: {e}")

ValueError: Can't convert non-rectangular Python sequence to Tensor.




En su lugar, cree un tf.RaggedTensor usando tf.ragged.constant :


In [71]:
ragged_t = tf.ragged.constant(ragged_list)

print(ragged_t.shape)

(4, None)


Tensores de strings

In [72]:
st = tf.constant("Este tensor string")
print(st)

tf.Tensor(b'Este tensor string', shape=(), dtype=string)


In [73]:
st = tf.constant(["Este tensor string",
                 "Cadena 2",
                 "Cadena 3",
                 "🥳"])
print(st)

tf.Tensor([b'Este tensor string' b'Cadena 2' b'Cadena 3' b'\xf0\x9f\xa5\xb3'], shape=(4,), dtype=string)


In [74]:
print(tf.strings.split(st))

<tf.RaggedTensor [[b'Este', b'tensor', b'string'], [b'Cadena', b'2'], [b'Cadena', b'3'], [b'\xf0\x9f\xa5\xb3']]>


In [75]:


st_split = tf.strings.split(st)
for i in st_split:
    print(i)



tf.Tensor([b'Este' b'tensor' b'string'], shape=(3,), dtype=string)
tf.Tensor([b'Cadena' b'2'], shape=(2,), dtype=string)
tf.Tensor([b'Cadena' b'3'], shape=(2,), dtype=string)
tf.Tensor([b'\xf0\x9f\xa5\xb3'], shape=(1,), dtype=string)


In [76]:
### string to number
st = tf.constant("1 10 10.4")

print(tf.strings.to_number(tf.strings.split(st, " ")))

tf.Tensor([ 1.  10.  10.4], shape=(3,), dtype=float32)



Tensores dispersos. SparseTensor


In [77]:
#  tensor disperso
sparse_tensor = tf.sparse.SparseTensor(indices = [[0,1], [1,2]],
                                       values = [1,2],
                                       dense_shape
                                        =[3,4])
print(sparse_tensor, "\n")

# convierte a tensor denso
print(tf.sparse.to_dense(sparse_tensor))

SparseTensor(indices=tf.Tensor(
[[0 1]
 [1 2]], shape=(2, 2), dtype=int64), values=tf.Tensor([1 2], shape=(2,), dtype=int32), dense_shape=tf.Tensor([3 4], shape=(2,), dtype=int64)) 

tf.Tensor(
[[0 1 0 0]
 [0 0 2 0]
 [0 0 0 0]], shape=(3, 4), dtype=int32)



zip

In [78]:
a = tf.data.Dataset.range(1,4) # ==> [1, 2, 3]
b = tf.data.Dataset.range(4,7) # ==> [4, 5, 6]

ds = tf.data.Dataset.zip((a, b))

In [79]:
for i in a:
    print(i.numpy())

1
2
3


In [80]:
for i in ds:
    print(i)

(<tf.Tensor: shape=(), dtype=int64, numpy=1>, <tf.Tensor: shape=(), dtype=int64, numpy=4>)
(<tf.Tensor: shape=(), dtype=int64, numpy=2>, <tf.Tensor: shape=(), dtype=int64, numpy=5>)
(<tf.Tensor: shape=(), dtype=int64, numpy=3>, <tf.Tensor: shape=(), dtype=int64, numpy=6>)


In [81]:
list(ds.as_numpy_iterator())

[(1, 4), (2, 5), (3, 6)]

In [82]:
c = tf.data.Dataset.range(7,11) # ==> [7, 8, 9, 10]

In [83]:


ds1 = tf.data.Dataset.zip((c,b,a))
list(ds1.as_numpy_iterator())



[(7, 4, 1), (8, 5, 2), (9, 6, 3)]